# 02 — CSD Detection and Validation

This notebook runs the two-layer Critical Slowing Down detector on the per-character SEP trajectories built in Notebook 1, manually validates the 20 detected segments against a literature-grounded codebook, and reproduces Figures 1–3 of the report.

**Reading order:**  `01_data_and_vad` → `02_csd_detection_and_validation` → `03_case_study_scene9`

## Setup

In [ ]:
import os, sys
NB_DIR = os.path.abspath('.')
sys.path.insert(0, os.path.join(NB_DIR, '..', 'code', 'model'))

import numpy as np
from IPython.display import Image

## 1. The two-layer detector

$$\mathrm{SEP}_t = V_t + 0.5 \cdot D_t$$

**Layer 1 — sustained negative deviation:**

$$\bar{\Delta}_t < -0.5\,\sigma_c$$

**Layer 2 — joint CSD elevation:**

$$r_1(t) > Q_{75}(r_1) \ \wedge\  \mathrm{Var}(t) > Q_{75}(\mathrm{Var})$$

with a one-step lead-lag tolerance so Layer 2 may fire just before Layer 1.  A segment fires only when both conditions hold for ≥3 consecutive same-episode utterances.  See `METHOD.md` for the full specification.

In [ ]:
from data_loader import load_and_sort, filter_speakers
from vad_engine import load_vad_lexicon, build_trajectories
from csd_detector import detect_csd_collapse
from config import JSON_PATHS, VAD_LEXICON_PATH, MAIN_CHARACTERS
from config_csd import (EWS_WINDOW, K_SIGMA, LEAD_LAG,
                        AC1_THRESHOLD_QUANTILE, VAR_THRESHOLD_QUANTILE,
                        MIN_SEGMENT_LEN)

print('CSD detector parameters:')
print(f'  k                  = {EWS_WINDOW}     (rolling window length)')
print(f'  k_sigma            = {K_SIGMA}    (Layer 1 threshold)')
print(f'  AC(1) quantile     = {AC1_THRESHOLD_QUANTILE}')
print(f'  Variance quantile  = {VAR_THRESHOLD_QUANTILE}')
print(f'  lead-lag tolerance = {LEAD_LAG}')
print(f'  min segment length = {MIN_SEGMENT_LEN}')

## 2. Run the detector on all six characters

In [ ]:
all_utts = filter_speakers(load_and_sort(JSON_PATHS))
vad_lex, mwe_lex = load_vad_lexicon(VAD_LEXICON_PATH)
trajs = build_trajectories(all_utts, vad_lex, mwe_lex, MAIN_CHARACTERS)

results = {}
for char in MAIN_CHARACTERS:
    segments, info = detect_csd_collapse(
        trajs[char], k=EWS_WINDOW, k_sigma=K_SIGMA,
        ac1_q=AC1_THRESHOLD_QUANTILE, var_q=VAR_THRESHOLD_QUANTILE,
        lead_lag=LEAD_LAG, min_len=MIN_SEGMENT_LEN,
    )
    results[char] = (segments, info)

n_total = sum(len(s) for s, _ in results.values())
print(f'\nTotal candidate tipping segments: {n_total}')
for char in MAIN_CHARACTERS:
    print(f'  {char:<18} {len(results[char][0])}')

## 3. Joey's full Seasons 1–4 trajectory  ·  Figure 1

Three pipeline-flagged segments — one true positive (SEG14, *The One Where Dr. Ramoray Dies*, S02E18) and two false positives.  The right-column inset zooms on SEG14 and shows the textbook CSD early-warning signature: $r_1$ and variance both rise *before* the SEP nadir.

In [ ]:
%run ../code/figures/build_fig1_joey_annotated.py
Image('../figures/fig1_joey_annotated.png')

## 4. Manual validation against the codebook  ·  Figure 2

All 20 segments were reviewed against criteria drawn from Beck's cognitive triad, Nolen-Hoeksema's rumination markers, Rude's first-person-pronoun analysis, Al-Mosaiwi's absolutist words, and Bowlby's protest–despair temporal arc (full codebook in `METHOD.md`).

Aggregate precision is **6 / 20 = 30 %**.  The 14 false positives partition into six recurring failure modes — the most informative is **emotion attribution error** (Notebook 3).

In [ ]:
%run ../code/figures/build_fig2_precision_audit.py
Image('../figures/fig2_precision_audit.png')

## 5. Per-character potential landscapes  ·  Figure 3

Empirical potential $U_c(\mathrm{SEP}) = -\log p_c(\mathrm{SEP})$, one panel per character, with TP segments (gold star) and FP segments (cross) placed at their mean SEP.  Each panel title shows per-character precision.

**Phoebe Buffay's 0/3 result** is the sharpest illustration of the structural problem: her three flagged segments sit in the dynamically right place on her own potential surface — and yet *none* survives manual review, because the scenes belong to other characters.

In [ ]:
%run ../code/figures/build_fig3_potential_curves.py
Image('../figures/fig3_potential_curves.png')

## 6. The structural error

Across the 14 false positives, four share a single mechanism: the target speaker shows negative-valence language not because they themselves are tipping, but because they are sympathetically engaging with another character who is.  The detector catches the linguistic signal but mis-locates its source.

This is **emotion attribution error**.  It is not a tunable parameter problem — no threshold change reassigns the signal to the right speaker, because at the lexical level the comforter's and the protagonist's vocabularies overlap.  Resolving it would require operating on the conversation, not on each speaker in isolation.

Notebook 3 makes this concrete on a single 24-utterance scene.